# Phase 3 — Models

**Purpose:** Fit three logistic regression models to the same data and compare them directly.

1. **No pooling:** Each segment gets an independent intercept. Small segments have high variance.
2. **Full pooling:** Single global intercept. Ignores segment membership entirely.
3. **Partial pooling (target):** Segment intercepts drawn from a shared population distribution with hyperparameters estimated from data.

All models use `cores=1` (required on WSL2/Windows) and non-centred parameterisation for the hierarchical model.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import arviz as az
import pymc as pm
import matplotlib.pyplot as plt

from src.bcs.models import fit_no_pooling, fit_full_pooling, fit_partial_pooling, check_divergences

print("Imports loaded successfully.")

## 1. Load prepared panel

In [ ]:
import pandas as pd

panel = pd.read_parquet("../data/customer_panel.parquet")
print(f"Panel loaded: {len(panel):,} customers, {panel['segment_idx'].nunique()} segments")

outcome = panel["churned"].values
segment_idx = panel["segment_idx"].values
n_segments = panel["segment_idx"].nunique()
print(f"Outcome shape: {outcome.shape}, Churn rate: {outcome.mean():.1%}")
print(f"Segments: {n_segments}")

## 2. Model A — No pooling

Each segment gets an independent intercept estimated only from its own data.
Small segments have high variance; no information is shared between segments.

$$\text{logit}(p_i) = \alpha_{j[i]}$$
$$\alpha_j \sim \mathcal{N}(0, 10)$$

In [ ]:
model_no_pool, trace_no_pool = fit_no_pooling(outcome, segment_idx, n_segments)
check_divergences(trace_no_pool)

## 3. Model B — Full pooling

Single global intercept. Segment membership is ignored entirely.

$$\text{logit}(p_i) = \alpha$$
$$\alpha \sim \mathcal{N}(0, 10)$$

In [ ]:
model_full, trace_full = fit_full_pooling(outcome)
check_divergences(trace_full)

## 4. Model C — Partial pooling (target model)

Segment intercepts are drawn from a shared population distribution.
The hyperparameters $\mu$ and $\tau$ are estimated from data.

$$\text{logit}(p_i) = \alpha_{j[i]}$$
$$\alpha_j \sim \mathcal{N}(\mu, \tau)$$
$$\mu \sim \mathcal{N}(0, 1)$$
$$\tau \sim \text{HalfNormal}(1)$$

Non-centred parameterisation: We use `alpha_offset` (standard normal) multiplied by `tau` rather than sampling `alpha` directly from `Normal(mu, tau)`. The reason is sampling efficiency with small tau, as with centered parameterization, NUTS can develop funnel geometry causing divergences.

In [ ]:
model_partial, trace_partial = fit_partial_pooling(outcome, segment_idx, n_segments)
check_divergences(trace_partial)

## 5. Divergence diagnostics

We had 8 divergences in the partial pooling model. Let's investigate whether they indicate a real problem or just slightly awkward sampling geometry.

**What we're looking for:**
- The parallel plot shows which parameter combinations coincide with divergent transitions.
- If divergences cluster at low `tau` values in the pair plot, this is classic funnel geometry — the posterior has a narrow funnel near `tau ≈ 0` where NUTS loses its footing.
- This is the most common cause in hierarchical models and is not a sign of model misspecification, just a sampling geometry problem.

In [ ]:
# 1. Where in parameter space are the divergences?
az.plot_parallel(trace_partial, var_names=["mu", "tau"], divergences=True)
plt.tight_layout()
plt.show()
print("Parallel plot rendered.")

In [ ]:
# 2. Is tau the culprit? Divergences near zero tau = funnel geometry
az.plot_pair(trace_partial, var_names=["tau", "mu"], divergences=True, figsize=(8, 8))
plt.tight_layout()
plt.show()
print("Pair plot rendered.")

In [ ]:
# 3. Summary — check r_hat and ess
summary = az.summary(trace_partial, var_names=["mu", "tau", "alpha"], round_to=3)
print(summary.to_string())

In [ ]:
# 4. Energy diagnostic — should be close to 1.0
az.plot_energy(trace_partial)
plt.tight_layout()
plt.show()
print("Energy plot rendered.")

### Interpretation

**Report back:**
- The `r_hat` values for `mu`, `tau`, and each `alpha` (all should be <1.01 for a clean chain)
- The `ess_bulk` values (effective sample size — should be >400 for 1000 draws)
- Whether divergences cluster at low `tau` in the pair plot

**The deeper question:** Are 8 divergences actually biasing our posterior, or just indicating slightly awkward geometry that the chain navigated correctly most of the time? If `r_hat < 1.01` and `ess_bulk > 400` everywhere, the 8 divergences are cosmetically bad but substantively harmless. If `r_hat > 1.01` for `tau` specifically, the chain has not mixed and the posterior is not trustworthy.

**Likely fix:** Since we used non-centred parameterisation (which already mitigates the funnel), `target_accept` is the lever to pull. Higher `target_accept` forces NUTS to take smaller steps, resolving most funnel-geometry divergences at the cost of slower sampling.

In [ ]:
# Resample with higher target_accept if diagnostics suggest it's needed
# Uncomment and run if r_hat > 1.01 or ess_bulk < 400:
#
# with model_partial:
#     trace_partial_v2 = pm.sample(
#         1000, tune=1500,
#         target_accept=0.95,
#         cores=1, random_seed=42,
#         progressbar=True
#     )
# check_divergences(trace_partial_v2)
# print(az.summary(trace_partial_v2, var_names=["mu", "tau"]))
print("Resampling cell ready if needed.")

## 6. Model diagnostics

In [ ]:
summary = az.summary(trace_partial, var_names=["mu", "tau"], hdi_prob=0.94)
print(summary.to_string())

In [ ]:
az.plot_trace(trace_partial, var_names=["mu", "tau"])
plt.tight_layout()
plt.show()
print("Trace plots rendered.")

In [ ]:
az.plot_posterior(trace_partial, var_names=["tau"], hdi_prob=0.94)
plt.tight_layout()
plt.show()
tau_samples = trace_partial.posterior["tau"].values.flatten()
tau_median = np.median(tau_samples)
tau_hdi = az.hdi(trace_partial, var_names=["tau"], hdi_prob=0.94)
print(f"tau posterior median: {tau_median:.3f}")
print(f"tau 94% HDI: [{tau_hdi['tau'].hdi_lower.values[0]:.3f}, {tau_hdi['tau'].hdi_high.values[0]:.3f}]")
if tau_median < 0.5:
    print("Interpretation: modest heterogeneity — partial pooling reduces to near full pooling.")
elif tau_median < 2:
    print("Interpretation: meaningful heterogeneity — segments differ on log-odds scale.")
else:
    print("Interpretation: strong heterogeneity — segments are nearly independent.")

## 7. Save traces

In [ ]:
trace_no_pool.to_netcdf("../data/trace_no_pool.nc")
trace_full.to_netcdf("../data/trace_full.nc")
trace_partial.to_netcdf("../data/trace_partial.nc")
print("All traces saved. Ready for Phase 4 (results).")